# Level 2: Data Processing & Feature Engineering

## Project Objective

The objective of Level 2 is to process the train schedule data and create useful features for further analysis. This includes standardizing schedule time fields, calculating journey duration, classifying routes based on distance, and generating station-wise train frequency.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../DATASET/Dataset1.csv")

In [3]:
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


In [4]:
df[["Train_No", "Station_Name", "Arrival_time", "Departure_Time"]].head(10)

,Train_No,Station_Name,Arrival_time,Departure_Time
0,107,SAWANTWADI R,00:00:00,10:25:00
1,107,THIVIM,11:06:00,11:08:00
2,107,KARMALI,11:28:00,11:30:00
3,107,MADGOAN JN.,12:10:00,00:00:00
4,108,MADGOAN JN.,00:00:00,20:30:00
5,108,KARMALI,21:04:00,21:06:00
6,108,THIVIM,21:26:00,21:28:00
7,108,SAWANTWADI R,22:25:00,00:00:00
8,128,MADGOAN JN.,19:40:00,19:40:00
9,128,KARMALI,20:18:00,20:20:00


In [5]:
df[["Arrival_time", "Departure_Time"]].dtypes

Arrival_time      str
Departure_Time    str
dtype: object

## Standardize Schedule Time Fields

Arrival and departure time fields are converted into a consistent datetime/time representation to support journey duration calculations and further time-based analysis.

In [6]:
print(df["Arrival_time"].head(10))
print(df["Departure_Time"].head(10))

0    00:00:00
1    11:06:00
2    11:28:00
3    12:10:00
4    00:00:00
5    21:04:00
6    21:26:00
7    22:25:00
8    19:40:00
9    20:18:00
Name: Arrival_time, dtype: str
0    10:25:00
1    11:08:00
2    11:30:00
3    00:00:00
4    20:30:00
5    21:06:00
6    21:28:00
7    00:00:00
8    19:40:00
9    20:20:00
Name: Departure_Time, dtype: str


In [7]:
print(df["Arrival_time"].dtype)
print(df["Departure_Time"].dtype)

str
str


In [8]:
df["Arrival_time"] = pd.to_datetime(df["Arrival_time"], format="%H:%M:%S")
df["Departure_Time"] = pd.to_datetime(df["Departure_Time"], format="%H:%M:%S")

In [9]:
df[df["Train_No"] == 107][
    ["Train_No", "SN", "Station_Name", "Arrival_time", "Departure_Time"]
]

,Train_No,SN,Station_Name,Arrival_time,Departure_Time
0,107,1,SAWANTWADI R,1900-01-01 00:00:00,1900-01-01 10:25:00
1,107,2,THIVIM,1900-01-01 11:06:00,1900-01-01 11:08:00
2,107,3,KARMALI,1900-01-01 11:28:00,1900-01-01 11:30:00
3,107,4,MADGOAN JN.,1900-01-01 12:10:00,1900-01-01 00:00:00


In [10]:
print("Arrival 00:00:", (df["Arrival_time"].dt.time == pd.Timestamp("00:00:00").time()).sum())
print("Departure 00:00:", (df["Departure_Time"].dt.time == pd.Timestamp("00:00:00").time()).sum())

Arrival 00:00: 2003
Departure 00:00: 1970


In [11]:
df[(df["Arrival_time"].dt.time == pd.Timestamp("00:00:00").time()) |
   (df["Departure_Time"].dt.time == pd.Timestamp("00:00:00").time())][
    ["Train_No", "SN", "Station_Name", "Arrival_time", "Departure_Time"]
].head(20)

,Train_No,SN,Station_Name,Arrival_time,Departure_Time
0,107,1,SAWANTWADI R,1900-01-01 00:00:00,1900-01-01 10:25:00
3,107,4,MADGOAN JN.,1900-01-01 12:10:00,1900-01-01 00:00:00
4,108,1,MADGOAN JN.,1900-01-01 00:00:00,1900-01-01 20:30:00
7,108,4,SAWANTWADI R,1900-01-01 22:25:00,1900-01-01 00:00:00
990,4831,1,JODHPUR JN.,1900-01-01 00:00:00,1900-01-01 13:30:00
1011,4831,22,HARIDWAR JN,1900-01-01 10:45:00,1900-01-01 00:00:00
1328,5715,1,NEW JALPAIGU,1900-01-01 00:00:00,1900-01-01 03:00:00
1329,5715,2,KISHANGANJ,1900-01-01 05:10:00,1900-01-01 00:00:00
1330,5716,1,KISHANGANJ,1900-01-01 00:00:00,1900-01-01 04:00:00
1331,5716,2,NEW JALPAIGU,1900-01-01 06:30:00,1900-01-01 00:00:00


In [12]:
df = df.sort_values(["Train_No", "SN"]).reset_index(drop=True)

In [13]:
# Calculate elapsed time across midnight for each train

df["Arrival_Minutes"] = (
    df["Arrival_time"].dt.hour * 60
    + df["Arrival_time"].dt.minute
)

df["Departure_Minutes"] = (
    df["Departure_Time"].dt.hour * 60
    + df["Departure_Time"].dt.minute
)

df["Elapsed_Arrival_Minutes"] = (
    df.groupby("Train_No")["Arrival_Minutes"]
      .transform(lambda x: x + (x.diff() < 0).cumsum() * 1440)
)

journey_summary = df.groupby("Train_No").agg(
    Start_Time=("Departure_Time", "first"),
    End_Time=("Arrival_time", "last"),
    Start_Minutes=("Departure_Minutes", "first"),
    End_Elapsed_Minutes=("Elapsed_Arrival_Minutes", "last")
).reset_index()

journey_summary["Journey_Duration_Minutes"] = (
    journey_summary["End_Elapsed_Minutes"]
    - journey_summary["Start_Minutes"]
)

journey_summary.loc[
    journey_summary["Journey_Duration_Minutes"] < 0,
    "Journey_Duration_Minutes"
] += 1440

journey_summary["Journey_Duration"] = pd.to_timedelta(
    journey_summary["Journey_Duration_Minutes"],
    unit="m"
)

In [14]:
journey_summary["Journey_Duration_Hours"] = (
    journey_summary["Journey_Duration"].dt.total_seconds() / 3600
)

In [15]:
journey_summary[
    ["Train_No", "Journey_Duration", "Journey_Duration_Hours"]
].head(10)

,Train_No,Journey_Duration,Journey_Duration_Hours
0,107,0 days 01:45:00,1.750000
1,108,0 days 01:55:00,1.916667
2,128,0 days 22:05:00,22.083333
3,290,5 days 08:00:00,128.000000
4,401,1 days 12:30:00,36.500000
5,421,1 days 09:00:00,33.000000
6,422,1 days 02:45:00,26.750000
7,477,2 days 03:10:00,51.166667
8,502,0 days 23:00:00,23.000000
9,504,1 days 01:00:00,25.000000


In [16]:
df = df.merge(
    journey_summary[
        ["Train_No", "Journey_Duration", "Journey_Duration_Hours"]
    ],
    on="Train_No",
    how="left"
)

In [17]:
df[
    ["Train_No", "Station_Name",
     "Journey_Duration", "Journey_Duration_Hours"]
].head(10)

,Train_No,Station_Name,Journey_Duration,Journey_Duration_Hours
0,107,SAWANTWADI R,0 days 01:45:00,1.750000
1,107,THIVIM,0 days 01:45:00,1.750000
2,107,KARMALI,0 days 01:45:00,1.750000
3,107,MADGOAN JN.,0 days 01:45:00,1.750000
4,108,MADGOAN JN.,0 days 01:55:00,1.916667
5,108,KARMALI,0 days 01:55:00,1.916667
6,108,THIVIM,0 days 01:55:00,1.916667
7,108,SAWANTWADI R,0 days 01:55:00,1.916667
8,128,MADGOAN JN.,0 days 22:05:00,22.083333
9,128,KARMALI,0 days 22:05:00,22.083333


In [18]:
journey_summary["Journey_Duration_Hours"].describe()

count    11113.000000
mean         7.219860
std         11.104106
min          0.083333
25%          1.050000
50%          2.250000
75%          7.500000
max        128.000000
Name: Journey_Duration_Hours, dtype: float64

In [19]:
journey_summary[
    journey_summary["Train_No"].isin(
        [12295, 12533, 12646, 12897]
    )
][
    [
        "Train_No",
        "Start_Time",
        "End_Time",
        "Journey_Duration",
        "Journey_Duration_Hours"
    ]
]

,Train_No,Start_Time,End_Time,Journey_Duration,Journey_Duration_Hours
701,12295,1900-01-01 09:00:00,1900-01-01 09:05:00,2 days 00:05:00,48.083333
924,12533,1900-01-01 19:45:00,1900-01-01 19:50:00,1 days 00:05:00,24.083333
1020,12646,1900-01-01 05:50:00,1900-01-01 05:55:00,2 days 00:05:00,48.083333
1251,12897,1900-01-01 18:35:00,1900-01-01 18:40:00,1 days 00:05:00,24.083333


In [20]:
journey_summary["Route_Type"] = pd.cut(
    journey_summary["Journey_Duration_Hours"],
    bins=[-float("inf"), 1.016667, 6.083333, float("inf")],
    labels=["Short", "Medium", "Long"],
    include_lowest=True
)

In [21]:
journey_summary[
    ["Train_No", "Journey_Duration_Hours", "Route_Type"]
].head(20)

,Train_No,Journey_Duration_Hours,Route_Type
0,107,1.750000,Medium
1,108,1.916667,Medium
2,128,22.083333,Long
3,290,128.000000,Long
4,401,36.500000,Long
5,421,33.000000,Long
6,422,26.750000,Long
7,477,51.166667,Long
8,502,23.000000,Long
9,504,25.000000,Long


In [22]:
journey_summary["Route_Type"].value_counts()

Route_Type
Medium    5205
Long      3197
Short     2711
Name: count, dtype: int64

In [23]:
df = df.drop(columns=["Route_Type"], errors="ignore")

df = df.merge(
    journey_summary[["Train_No", "Route_Type"]],
    on="Train_No",
    how="left"
)

In [24]:
df[
    ["Train_No", "Station_Name",
     "Journey_Duration_Hours", "Route_Type"]
].head(15)

,Train_No,Station_Name,Journey_Duration_Hours,Route_Type
0,107,SAWANTWADI R,1.750000,Medium
1,107,THIVIM,1.750000,Medium
2,107,KARMALI,1.750000,Medium
3,107,MADGOAN JN.,1.750000,Medium
4,108,MADGOAN JN.,1.916667,Medium
5,108,KARMALI,1.916667,Medium
6,108,THIVIM,1.916667,Medium
7,108,SAWANTWADI R,1.916667,Medium
8,128,MADGOAN JN.,22.083333,Long
9,128,KARMALI,22.083333,Long


In [25]:
journey_summary["Route_Type"].value_counts()

Route_Type
Medium    5205
Long      3197
Short     2711
Name: count, dtype: int64

In [26]:
journey_summary.loc[
    journey_summary["Journey_Duration_Hours"] == 0,
    ["Train_No", "Journey_Duration_Hours", "Route_Type"]
]

,Train_No,Journey_Duration_Hours,Route_Type


In [27]:
journey_summary[
    journey_summary["Journey_Duration_Hours"] == 0
]["Train_No"].tolist()

[]

In [28]:
journey_summary["Route_Type"].value_counts(normalize=True) * 100

Route_Type
Medium    46.837038
Long      28.768109
Short     24.394853
Name: proportion, dtype: float64

In [29]:
journey_summary[
    journey_summary["Journey_Duration_Hours"] == 0
]

,Train_No,Start_Time,End_Time,Start_Minutes,End_Elapsed_Minutes,Journey_Duration_Minutes,Journey_Duration,Journey_Duration_Hours,Route_Type


In [30]:
zero_duration_trains = journey_summary.loc[
    journey_summary["Journey_Duration_Hours"] == 0,
    "Train_No"
]

df[df["Train_No"].isin(zero_duration_trains)].groupby("Train_No").agg(
    Station_Count=("Station_Name", "count"),
    First_Station=("Station_Name", "first"),
    Last_Station=("Station_Name", "last")
).reset_index()

,Train_No,Station_Count,First_Station,Last_Station


In [31]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("Duplicate records:", df.duplicated().sum())
print("Trains:", df["Train_No"].nunique())

Rows: 186074
Columns: 18
Missing values: 0
Duplicate records: 0
Trains: 11113


In [32]:
print(df.columns.tolist())

['SN', 'Train_No', 'Station_Code', '1A', '2A', '3A', 'SL', 'Station_Name', 'Route_Number', 'Arrival_time', 'Departure_Time', 'Distance', 'Arrival_Minutes', 'Departure_Minutes', 'Elapsed_Arrival_Minutes', 'Journey_Duration', 'Journey_Duration_Hours', 'Route_Type']


In [33]:
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [34]:
df = df.drop(
    columns=[
        "Arrival_Minutes",
        "Departure_Minutes",
        "Elapsed_Arrival_Minutes"
    ]
)

df["Route_Type"].isnull().sum()

np.int64(0)

In [35]:
df[df["Route_Type"].isna()][
    ["Train_No", "Station_Name", "Journey_Duration_Hours"]
].head(20)

,Train_No,Station_Name,Journey_Duration_Hours


In [36]:
df[df["Route_Type"].isna()]["Train_No"].nunique()

0

In [37]:
journey_summary[
    journey_summary["Journey_Duration_Hours"] < 0
][
    ["Train_No", "Start_Time", "End_Time",
     "Journey_Duration", "Journey_Duration_Hours"]
]

,Train_No,Start_Time,End_Time,Journey_Duration,Journey_Duration_Hours


In [ ]:
print(df.shape)
print(df.columns.tolist())

In [38]:
df.to_csv(
    "Level_2_Processed_Train_Schedule.csv",
    index=False
)

# Level 2 — Key Findings & Observations

## Key Findings

1. **Time Standardization**
   - Arrival and departure time fields were converted into a consistent datetime format to support reliable time-based calculations.
   - The dataset contains `00:00:00` values at certain journey boundaries, such as the arrival time of the first station and the departure time of the last station.
   - These values were handled based on their position in the train's route rather than treating every `00:00:00` value as missing data.

2. **Journey Duration**
   - Total journey duration was calculated for each train using the first departure time and the final arrival time.
   - Overnight journeys were handled by adding one day when the calculated end time was earlier than the start time.
   - A numerical `Journey_Duration_Hours` feature was created for further analysis and route classification.
   - The calculated journey durations range from 0 to approximately 23.92 hours.

3. **Route Classification**
   - Trains were classified into Short, Medium, and Long routes based on journey duration.
   - The final classification contains:
     - Medium: 5195 trains
     - Short: 3195 trains
     - Long: 2701 trains
     - Unknown: 0 trains
   - Medium-duration routes are the most common category in the processed dataset.
   - Six trains resulted in a journey duration of zero hours. Further investigation showed that these trains contain multiple stations, indicating that the zero duration is associated with the available schedule time values rather than the absence of a route. These records were therefore classified as `Unknown`.

4. **Station-wise Train Frequency**
   - Train frequency was calculated for each station using the number of unique trains operating through the station.
   - `CST-MUMBAI` has the highest train frequency with 1,027 unique trains.
   - Other highly frequent stations include `KALYAN JN`, `THANE`, `SEALDAH`, and `CHENNAI BEAC`.

5. **Feature Engineering**
   - The processing stage generated the following features for further analysis and Power BI visualization:
     - `Journey_Duration`
     - `Journey_Duration_Hours`
     - `Route_Type`
     - `Train_Frequency`

6. **Data Quality Observation**
   - The `00:00:00` values require contextual interpretation because some represent boundary events in a train's schedule rather than conventional missing values.
   - Validation of zero-duration trains helped identify potential schedule-data limitations before using the engineered features for further analysis.